In [ ]:
#!/usr/bin/env python3
"""
VISUALIZE STAFFING CHANGE (NURSES + DOCTORS) PRE VS PANDEMIC
- By deprivation group (Low / Mid / High), across years (2018â€“2022)
- Produces:
  (1) Unweighted (hospital-equal) trends + pre/post bars
  (2) Respondent-weighted trends + pre/post bars (weights = nobs)
  (3) Combined comparison plots (unweighted vs respondent-weighted)

INPUT FILES (EDIT PATHS):
  1) hospital staff and hospitallos.xlsx
     Expected columns: IK, year, nurseper1kpatientday, mdper1kpatientday
     (If nurseper1kpatientday/mdper1kpatientday missing, script will compute from hospnurse/hospmd/hosplos if present)
  2) hosp_gisd_outputs_allinone.xlsx  (sheet: ik_all_years)
     Expected columns: IK, year, gisd_mean_15km
  3) PEQ number of observations by IK and year.xlsx
     Expected columns: IK, year, nobs

OUTPUT:
  out_staffing_weighted_viz/
    trend_nurses_unweighted.png
    trend_nurses_resp_weighted.png
    trend_nurses_combined.png
    trend_doctors_unweighted.png
    trend_doctors_resp_weighted.png
    trend_doctors_combined.png
    prepost_nurses_unweighted.png
    prepost_nurses_resp_weighted.png
    prepost_nurses_combined.png
    prepost_doctors_unweighted.png
    prepost_doctors_resp_weighted.png
    prepost_doctors_combined.png
    staffing_summary_tables.xlsx
"""

from __future__ import annotations
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =========================
# PATHS (EDIT THESE)
# =========================
STAFF_XLSX = r"C:\Users\Caleb\Downloads\viz2\hospital staff and hospitallos.xlsx"
GISD_XLSX  = r"C:\Users\Caleb\Downloads\viz2\hosp_gisd_outputs_allinone.xlsx"
RESP_XLSX  = r"C:\Users\Caleb\Downloads\viz2\PEQ number of observations by IK and year.xlsx"

OUT_DIR = "out_staffing_weighted_viz"

# =========================
# SETTINGS
# =========================
PANDEMIC_START_YEAR = 2020
PRE_YEARS = [2018, 2019]
PANDEMIC_YEARS = [2020, 2021, 2022]

# If higher GISD means MORE deprivation, keep True.
# If your GISD is reversed, set False.
HIGHER_GISD_MORE_DEPRIVED = True

DEPRIVATION_ORDER = ["Low deprivation", "Mid deprivation", "High deprivation"]

# =========================
# HELPERS
# =========================
def _require_cols(df: pd.DataFrame, cols: list[str], label: str) -> None:
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"{label}: missing columns {missing}. Found columns: {list(df.columns)}")

def _as_int(df: pd.DataFrame, col: str) -> pd.DataFrame:
    df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    return df

def load_staff(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    _require_cols(df, ["IK", "year"], "STAFF")
    df = _as_int(df, "IK")
    df = _as_int(df, "year")

    # Use existing per-1k rates if present; else compute if possible
    if "nurseper1kpatientday" not in df.columns:
        _require_cols(df, ["hospnurse", "hosplos"], "STAFF (to compute nurseper1kpatientday)")
        df["nurseper1kpatientday"] = np.where(df["hosplos"] > 0, (df["hospnurse"] / df["hosplos"]) * 1000.0, np.nan)

    if "mdper1kpatientday" not in df.columns:
        _require_cols(df, ["hospmd", "hosplos"], "STAFF (to compute mdper1kpatientday)")
        df["mdper1kpatientday"] = np.where(df["hosplos"] > 0, (df["hospmd"] / df["hosplos"]) * 1000.0, np.nan)

    keep = ["IK", "year", "nurseper1kpatientday", "mdper1kpatientday"]
    # keep optional raw cols if present (useful for debugging)
    for c in ["hosplos", "hospnurse", "hospmd", "hospsmd"]:
        if c in df.columns:
            keep.append(c)

    return df[keep].copy()

def load_gisd(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name="ik_all_years")
    _require_cols(df, ["IK", "year", "gisd_mean_15km"], "GISD (ik_all_years)")
    df = _as_int(df, "IK")
    df = _as_int(df, "year")
    return df[["IK", "year", "gisd_mean_15km"]].copy()

def load_respondents(path: str) -> pd.DataFrame:
    df = pd.read_excel(path)
    _require_cols(df, ["IK", "year", "nobs"], "RESPONDENTS")
    df = _as_int(df, "IK")
    df = _as_int(df, "year")
    df["nobs"] = pd.to_numeric(df["nobs"], errors="coerce")
    df = df.dropna(subset=["IK", "year", "nobs"])
    return df

def assign_deprivation_groups(panel: pd.DataFrame) -> pd.DataFrame:
    """
    Stable grouping: compute hospital mean GISD across all years, then split into terciles.
    Uses rank->qcut to guarantee 3 groups even with many ties in GISD.
    """
    g = (panel.groupby("IK", as_index=False)["gisd_mean_15km"]
         .mean()
         .rename(columns={"gisd_mean_15km": "gisd_mean_15km_hospmean"}))

    g["rank"] = g["gisd_mean_15km_hospmean"].rank(method="first")
    g["tercile"] = pd.qcut(g["rank"], 3, labels=[1, 2, 3]).astype(int)

    if HIGHER_GISD_MORE_DEPRIVED:
        mapping = {1: "Low deprivation", 2: "Mid deprivation", 3: "High deprivation"}
    else:
        mapping = {1: "High deprivation", 2: "Mid deprivation", 3: "Low deprivation"}

    g["deprivation_group"] = g["tercile"].map(mapping)

    out = panel.merge(g[["IK", "deprivation_group", "gisd_mean_15km_hospmean"]], on="IK", how="left")
    out["deprivation_group"] = pd.Categorical(out["deprivation_group"], categories=DEPRIVATION_ORDER, ordered=True)
    return out

def add_period(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["period"] = np.where(df["year"].isin(PRE_YEARS), "Pre (2018â€“2019)",
                    np.where(df["year"].isin(PANDEMIC_YEARS), "Pandemic (2020â€“2022)", "Other"))
    return df

def weighted_avg(values: pd.Series, weights: pd.Series) -> float:
    v = pd.to_numeric(values, errors="coerce")
    w = pd.to_numeric(weights, errors="coerce")
    m = (~v.isna()) & (~w.isna()) & (w > 0)
    if m.sum() == 0:
        return np.nan
    return float(np.average(v[m].to_numpy(), weights=w[m].to_numpy()))

def yearly_unweighted(panel: pd.DataFrame) -> pd.DataFrame:
    out = (panel.groupby(["deprivation_group", "year"], as_index=False)
           .agg(
               hospitals=("IK", "nunique"),
               nurse_unweighted=("nurseper1kpatientday", "mean"),
               doctor_unweighted=("mdper1kpatientday", "mean"),
           ))
    return out

def yearly_resp_weighted(panel: pd.DataFrame) -> pd.DataFrame:
    out = (panel.groupby(["deprivation_group", "year"], as_index=False)
           .apply(lambda g: pd.Series({
               "hospitals_with_weights": int(g.dropna(subset=["nobs"])["IK"].nunique()),
               "respondents_sum": float(np.nansum(g["nobs"].to_numpy())),
               "nurse_resp_weighted": weighted_avg(g["nurseper1kpatientday"], g["nobs"]),
               "doctor_resp_weighted": weighted_avg(g["mdper1kpatientday"], g["nobs"]),
           }))
           .reset_index(drop=True))
    return out

def prepost_unweighted(panel: pd.DataFrame, metric: str) -> pd.DataFrame:
    df = add_period(panel)
    df = df[df["period"].isin(["Pre (2018â€“2019)", "Pandemic (2020â€“2022)"])].copy()
    out = (df.groupby(["deprivation_group", "period"], as_index=False)
           .agg(value=(metric, "mean")))
    return out

def prepost_resp_weighted(panel: pd.DataFrame, metric: str) -> pd.DataFrame:
    df = add_period(panel)
    df = df[df["period"].isin(["Pre (2018â€“2019)", "Pandemic (2020â€“2022)"])].copy()
    out = (df.groupby(["deprivation_group", "period"], as_index=False)
           .apply(lambda g: pd.Series({"value": weighted_avg(g[metric], g["nobs"])}))
           .reset_index(drop=True))
    return out

# =========================
# PLOTTING
# =========================
def plot_trend_single(df: pd.DataFrame, value_col: str, title: str, ylabel: str, outpath: str) -> None:
    plt.figure(figsize=(10, 6))
    for grp in DEPRIVATION_ORDER:
        d = df[df["deprivation_group"] == grp].sort_values("year")
        if len(d) == 0:
            continue
        plt.plot(d["year"], d[value_col], marker="o", label=grp)
    plt.axvline(PANDEMIC_START_YEAR, linewidth=1)
    plt.title(title)
    plt.xlabel("Year")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

def plot_trend_combined(unw: pd.DataFrame, w: pd.DataFrame, col_unw: str, col_w: str,
                        title: str, ylabel: str, outpath: str) -> None:
    plt.figure(figsize=(10, 6))
    for grp in DEPRIVATION_ORDER:
        d1 = unw[unw["deprivation_group"] == grp].sort_values("year")
        d2 = w[w["deprivation_group"] == grp].sort_values("year")
        if len(d1):
            plt.plot(d1["year"], d1[col_unw], marker="o", label=f"{grp} (unweighted)")
        if len(d2):
            plt.plot(d2["year"], d2[col_w], linestyle="--", marker="x", label=f"{grp} (respondent-weighted)")
    plt.axvline(PANDEMIC_START_YEAR, linewidth=1)
    plt.title(title)
    plt.xlabel("Year")
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

def plot_prepost_single(df: pd.DataFrame, title: str, ylabel: str, outpath: str) -> None:
    pivot = (df.pivot(index="deprivation_group", columns="period", values="value")
             .reindex(DEPRIVATION_ORDER))
    pivot.plot(kind="bar", figsize=(10, 6))
    plt.title(title)
    plt.xlabel("")
    plt.ylabel(ylabel)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

def plot_prepost_combined(unw: pd.DataFrame, w: pd.DataFrame,
                          title: str, ylabel: str, outpath: str) -> None:
    a = unw.copy()
    a["series"] = "Unweighted"
    b = w.copy()
    b["series"] = "Respondent-weighted"
    df = pd.concat([a, b], ignore_index=True)

    pivot = (df.pivot_table(index="deprivation_group", columns=["period", "series"], values="value", aggfunc="mean")
             .reindex(DEPRIVATION_ORDER))

    pivot.plot(kind="bar", figsize=(12, 6))
    plt.title(title)
    plt.xlabel("")
    plt.ylabel(ylabel)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

def plot_heatmap(matrix: np.ndarray, row_labels: list[str], col_labels: list[str],
                 title: str, outpath: str) -> None:
    # Adjust figure size based on number of rows
    height = max(8, 0.45 * len(row_labels))
    plt.figure(figsize=(10, height))
    im = plt.imshow(matrix, aspect="auto")
    plt.colorbar(im)
    # Set smaller font sizes for labels to fit all items
    plt.yticks(range(len(row_labels)), row_labels, fontsize=7)
    plt.xticks(range(len(col_labels)), col_labels, fontsize=9)
    plt.title(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(outpath, dpi=200)
    plt.close()

# =========================
# RUN
# =========================
def main() -> None:
    os.makedirs(OUT_DIR, exist_ok=True)

    staff = load_staff(STAFF_XLSX)
    gisd  = load_gisd(GISD_XLSX)
    resp  = load_respondents(RESP_XLSX)

    # Build panel
    panel = staff.merge(gisd, on=["IK", "year"], how="left")
    panel = assign_deprivation_groups(panel)
    panel = panel.merge(resp, on=["IK", "year"], how="left")

    # Drop rows with no deprivation group or no staffing rates
    panel = panel.dropna(subset=["deprivation_group", "nurseper1kpatientday", "mdper1kpatientday"])

    # Yearly summaries
    yw_unw = yearly_unweighted(panel)
    yw_w   = yearly_resp_weighted(panel)

    # ===== Nurses: yearly trends
    plot_trend_single(
        yw_unw, "nurse_unweighted",
        "Nurse staffing intensity (unweighted) by deprivation group (2018â€“2022)",
        "Nurses per 1,000 patient-days",
        os.path.join(OUT_DIR, "trend_nurses_unweighted.png"),
    )
    plot_trend_single(
        yw_w, "nurse_resp_weighted",
        "Nurse staffing intensity (respondent-weighted) by deprivation group (2018â€“2022)",
        "Nurses per 1,000 patient-days",
        os.path.join(OUT_DIR, "trend_nurses_resp_weighted.png"),
    )
    plot_trend_combined(
        yw_unw, yw_w, "nurse_unweighted", "nurse_resp_weighted",
        "Nurse staffing intensity: unweighted vs respondent-weighted (2018â€“2022)",
        "Nurses per 1,000 patient-days",
        os.path.join(OUT_DIR, "trend_nurses_combined.png"),
    )

    # ===== Doctors: yearly trends
    plot_trend_single(
        yw_unw, "doctor_unweighted",
        "Doctor staffing intensity (unweighted) by deprivation group (2018â€“2022)",
        "Doctors per 1,000 patient-days",
        os.path.join(OUT_DIR, "trend_doctors_unweighted.png"),
    )
    plot_trend_single(
        yw_w, "doctor_resp_weighted",
        "Doctor staffing intensity (respondent-weighted) by deprivation group (2018â€“2022)",
        "Doctors per 1,000 patient-days",
        os.path.join(OUT_DIR, "trend_doctors_resp_weighted.png"),
    )
    plot_trend_combined(
        yw_unw, yw_w, "doctor_unweighted", "doctor_resp_weighted",
        "Doctor staffing intensity: unweighted vs respondent-weighted (2018â€“2022)",
        "Doctors per 1,000 patient-days",
        os.path.join(OUT_DIR, "trend_doctors_combined.png"),
    )

    # ===== Pre vs Pandemic (Nurses)
    pp_n_unw = prepost_unweighted(panel, "nurseper1kpatientday")
    pp_n_w   = prepost_resp_weighted(panel, "nurseper1kpatientday")
    plot_prepost_single(
        pp_n_unw,
        "Pre vs pandemic nurse staffing (unweighted)",
        "Nurses per 1,000 patient-days",
        os.path.join(OUT_DIR, "prepost_nurses_unweighted.png"),
    )
    plot_prepost_single(
        pp_n_w,
        "Pre vs pandemic nurse staffing (respondent-weighted)",
        "Nurses per 1,000 patient-days",
        os.path.join(OUT_DIR, "prepost_nurses_resp_weighted.png"),
    )
    plot_prepost_combined(
        pp_n_unw, pp_n_w,
        "Pre vs pandemic nurse staffing: unweighted vs respondent-weighted",
        "Nurses per 1,000 patient-days",
        os.path.join(OUT_DIR, "prepost_nurses_combined.png"),
    )

    # ===== Pre vs Pandemic (Doctors)
    pp_d_unw = prepost_unweighted(panel, "mdper1kpatientday")
    pp_d_w   = prepost_resp_weighted(panel, "mdper1kpatientday")
    plot_prepost_single(
        pp_d_unw,
        "Pre vs pandemic doctor staffing (unweighted)",
        "Doctors per 1,000 patient-days",
        os.path.join(OUT_DIR, "prepost_doctors_unweighted.png"),
    )
    plot_prepost_single(
        pp_d_w,
        "Pre vs pandemic doctor staffing (respondent-weighted)",
        "Doctors per 1,000 patient-days",
        os.path.join(OUT_DIR, "prepost_doctors_resp_weighted.png"),
    )
    plot_prepost_combined(
        pp_d_unw, pp_d_w,
        "Pre vs pandemic doctor staffing: unweighted vs respondent-weighted",
        "Doctors per 1,000 patient-days",
        os.path.join(OUT_DIR, "prepost_doctors_combined.png"),
    )

    # ===== Export tables
    xlsx_out = os.path.join(OUT_DIR, "staffing_summary_tables.xlsx")
    with pd.ExcelWriter(xlsx_out, engine="openpyxl") as writer:
        panel.to_excel(writer, sheet_name="panel_hosp_year", index=False)
        yw_unw.to_excel(writer, sheet_name="yearly_unweighted", index=False)
        yw_w.to_excel(writer, sheet_name="yearly_resp_weighted", index=False)
        pp_n_unw.to_excel(writer, sheet_name="prepost_nurse_unw", index=False)
        pp_n_w.to_excel(writer, sheet_name="prepost_nurse_w", index=False)
        pp_d_unw.to_excel(writer, sheet_name="prepost_doctor_unw", index=False)
        pp_d_w.to_excel(writer, sheet_name="prepost_doctor_w", index=False)

if __name__ == "__main__":
    main()


In [ ]:
%pip install python-docx
